In [7]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 1
RO_CH = 0

gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
# -----------------------------------------------------------------------------
def best_n_samples(freq_hz, target_dur_s, sample_rate, samps_per_clk_, search_radius=3):
    target_n = int(round(target_dur_s * sample_rate))
    target_n = max(target_n, 1)
    pad0 = (-target_n) % samps_per_clk_
    target_n += pad0

    candidates = [target_n + k * samps_per_clk_
                  for k in range(-search_radius, search_radius + 1)]
    candidates = [n for n in candidates if n >= samps_per_clk_]

    if freq_hz == 0 or len(candidates) == 0:
        return target_n

    def residual(n):
        cycles = freq_hz * n / sample_rate
        frac = cycles - np.floor(cycles)
        return min(frac, 1.0 - frac)

    return min(candidates, key=residual)


def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0,
                    n_samples_override=None):
    if n_samples_override is not None:
        n_samples = max(1, int(n_samples_override))
    else:
        n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION
# -----------------------------------------------------------------------------
CHIRP_SPAN_HZ = 350e6
CHIRP_OFFSET_STOP_HZ = 0.0
CHIRP_OFFSET_START_HZ = CHIRP_OFFSET_STOP_HZ - CHIRP_SPAN_HZ

AMPLITUDE = 0.12

BASE_TONES_HZ = np.array([-76.25e6, 0, +122.92e6-76.25e6, +147.82e6-76.25e6])
TONE_RATIOS = np.array([0.337, 0.167, 0.288, 0.208])

BASE_TONES_HZ *= -1
NUM_TONES = len(BASE_TONES_HZ)

TOTAL_SWEEP_S_TARGET = 6e-3   # design target; actual value snaps slightly, see below

max_feasible_total_s = 0.95 * ENV_MAXLEN / ENV_SR
tone_fractions = TONE_RATIOS / TONE_RATIOS.sum()

NUM_STEPS = 46
print(f"NUM_STEPS = {NUM_STEPS}")

CYCLE_S = max_feasible_total_s / (NUM_STEPS + 3)   # nominal target for one composite buffer
tone_durations_s = tone_fractions * CYCLE_S

maxv = soccfg.get_maxv(GEN_CH)

# -----------------------------------------------------------------------------
# FIX A: FREEZE per-tone piece lengths once (evaluated at the final/trap
# frequency, since the trap buffer must line up phase-continuously with the
# last sweep step anyway) and reuse the SAME lengths for every chirp step
# and the trap buffer. This is required so every add_envelope() buffer has
# IDENTICAL total length: the tProc address-stepping trick in initialize()
# (mathi(r_addr, r_addr, '+', addr_step)) assumes a constant stride between
# consecutive envelope buffers. With the old per-step-optimized lengths
# (visible as a spread of 16 different buffer sizes), that stride was wrong
# for every step after the first -- the DAC read pointer silently drifted
# off the true buffer boundaries, worse with more steps. That is the actual
# cause of "phase continuity disappears as NUM_STEPS grows": it isn't a
# subtle phase-tracking error, it's the envelope pointer reading the wrong
# samples.
# -----------------------------------------------------------------------------
NOMINAL_PIECE_LENS = []
for base_tone, dur in zip(BASE_TONES_HZ, tone_durations_s):
    f0 = CHIRP_OFFSET_STOP_HZ + base_tone
    NOMINAL_PIECE_LENS.append(best_n_samples(f0, dur, ENV_SR, samps_per_clk))

samples_per_step = sum(NOMINAL_PIECE_LENS)   # now FIXED for every buffer, sweep + trap
assert samples_per_step % samps_per_clk == 0
assert min(NOMINAL_PIECE_LENS) >= 3 * samps_per_clk, (
    f"Smallest fixed tone slice is only {min(NOMINAL_PIECE_LENS)} samples "
    f"({min(NOMINAL_PIECE_LENS)/samps_per_clk:.1f} fabric cycles) -- hardware needs >= 3. "
    f"Reduce NUM_STEPS, or make TONE_RATIOS less extreme."
)

print("Fixed per-tone piece lengths (samples), used for every step + trap:")
for i, (bt, r, n) in enumerate(zip(BASE_TONES_HZ, TONE_RATIOS, NOMINAL_PIECE_LENS)):
    print(f"  tone {i} ({bt/1e6:+.1f} MHz offset): ratio {r} -> {n} samples "
          f"({n/samps_per_clk:.1f} fabric cycles)")
print(f"Composite buffer length (fixed, all steps + trap): {samples_per_step} samples")

# -----------------------------------------------------------------------------
# FIX B: snap the per-step HOLD TIME to an exact integer number of periods of
# the (now fixed-length) composite buffer. mode="periodic" free-runs the DAC
# at period T_buf = samples_per_step / ENV_SR; if the hold time isn't a whole
# multiple of T_buf, the retrigger into the next chirp step lands mid-cycle
# of the current buffer, and the phase actually present on the DAC at that
# instant no longer matches the phase the Python "phase" variable assumed
# when it built the next step's buffer. Forcing an integer number of periods
# K makes the retrigger always land exactly at a buffer-restart boundary, so
# the DAC's real phase at that instant matches what was used to seed the
# next step's serrodyne_tone(..., phase0=phase).
# -----------------------------------------------------------------------------
T_buf = samples_per_step / ENV_SR   # exact composite-buffer period, seconds

target_step_hold_s = TOTAL_SWEEP_S_TARGET / NUM_STEPS
K = max(1, int(round(target_step_hold_s / T_buf)))
STEP_HOLD_S = K * T_buf
STEP_HOLD_US = STEP_HOLD_S * 1e6
TOTAL_SWEEP_S = STEP_HOLD_S * NUM_STEPS   # actual, slightly snapped from the 6 ms target

print(f"Composite buffer period T_buf = {T_buf*1e9:.4f} ns")
print(f"K (periods held per chirp step) = {K}")
print(f"STEP_HOLD_US = {STEP_HOLD_US:.4f} us "
      f"(target was {target_step_hold_s*1e6:.4f} us, "
      f"snap delta = {(STEP_HOLD_S-target_step_hold_s)*1e9:+.3f} ns)")
print(f"Actual total sweep time = {TOTAL_SWEEP_S*1e3:.4f} ms "
      f"(target was {TOTAL_SWEEP_S_TARGET*1e3:.3f} ms)")
print("Note: sync_all()/us2cycles() still quantizes this wait to the tProc's own "
      "clock (a separate, generally much finer clock domain than the DAC sample "
      "clock), so there is a small residual rounding error on top of this -- but "
      "it is now at most ~1 tProc cycle instead of up to ~1 full buffer period.")

def build_multitone_buffer(chirp_offset_hz, phase0):
    """One composite envelope: all NUM_TONES tones, each shifted by the same
    chirp_offset_hz, concatenated. Every tone's piece length is taken from
    the FROZEN NOMINAL_PIECE_LENS so every buffer this function returns has
    the same total length regardless of chirp_offset_hz."""
    i_pieces, q_pieces = [], []
    phase = phase0
    for base_tone, dur, n_fixed in zip(BASE_TONES_HZ, tone_durations_s, NOMINAL_PIECE_LENS):
        f = chirp_offset_hz + base_tone
        y, phase, n_samples = serrodyne_tone(f, dur, ENV_SR, amplitude=AMPLITUDE, phase0=phase,
                                              n_samples_override=n_fixed)
        pad = (-len(y)) % samps_per_clk
        if pad:
            y = np.concatenate([y, np.zeros(pad)])
        i_pieces.append(np.round(y * maxv).astype(np.int16))
        q_pieces.append(np.zeros(len(y), dtype=np.int16))
    return np.concatenate(i_pieces), np.concatenate(q_pieces), phase, [len(p) for p in i_pieces]

# -----------------------------------------------------------------------------
# Midpoint chirp sampling (unchanged): each held step represents the chirp's
# average frequency over that dwell interval, halving peak phase error vs.
# edge sampling. Last step snaps exactly to CHIRP_OFFSET_STOP_HZ for a clean
# phase-continuous hand-off into the trap buffer.
# -----------------------------------------------------------------------------
step_width_hz = (CHIRP_OFFSET_STOP_HZ - CHIRP_OFFSET_START_HZ) / NUM_STEPS
chirp_offsets_hz = CHIRP_OFFSET_START_HZ + (np.arange(NUM_STEPS) + 0.5) * step_width_hz
chirp_offsets_hz[-1] = CHIRP_OFFSET_STOP_HZ

idata_list = []
qdata_list = []
phase = 0.0
for offset in chirp_offsets_hz:
    idata, qdata, phase, piece_lens = build_multitone_buffer(offset, phase)
    idata_list.append(idata)
    qdata_list.append(qdata)

lengths = set(len(x) for x in idata_list)
print(f"Distinct per-step buffer lengths: {lengths}  <- should now be a single value")
assert len(lengths) == 1, "Buffer lengths are still not uniform -- addressing will break."

total_samples = samples_per_step * NUM_STEPS
print(f"Per-step composite buffer length: {samples_per_step} samples "
      f"(tone slices: {piece_lens}, smallest = {min(piece_lens)/samps_per_clk:.1f} fabric cycles)")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
assert total_samples <= ENV_MAXLEN
assert samples_per_step % samps_per_clk == 0

# -----------------------------------------------------------------------------
# 3b. TRAP BUFFER: same fixed-length 4-tone composite, at the FINAL chirp
#     frequency, phase-continuous with the last sweep step.
# -----------------------------------------------------------------------------
trap_idata, trap_qdata, phase, trap_piece_lens = build_multitone_buffer(CHIRP_OFFSET_STOP_HZ, phase)
print(f"Trap buffer: {len(trap_idata)} samples (tone slices: {trap_piece_lens})")
assert len(trap_idata) == samples_per_step, "Trap buffer length must match sweep step length."

total_with_trap = total_samples + len(trap_idata)
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")
assert total_with_trap <= ENV_MAXLEN, (
    f"Sweep + trap buffers exceed memory: need {total_with_trap}, "
    f"have {ENV_MAXLEN}. Reduce NUM_STEPS or CYCLE_S."
)

# -----------------------------------------------------------------------------
# 4. PROGRAM
# -----------------------------------------------------------------------------
class RepeatedStepSerrodyneProgram(RAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for i, (idata_step, qdata_step) in enumerate(zip(cfg["idata_list"], cfg["qdata_list"])):
            self.add_envelope(ch=res_ch, name=f"serr_{i}", idata=idata_step, qdata=qdata_step)

        self.add_envelope(ch=res_ch, name="trap_wfm", idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])

        self.set_pulse_registers(
            ch=res_ch,
            style="arb",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            waveform="serr_0",
            outsel="input",
            mode="periodic",
        )

        self.r_rp = self.ch_page(res_ch)
        self.r_addr = self.sreg(res_ch, "addr")
        self.addr_step = cfg["samples_per_step"] // self.soccfg["gens"][res_ch]["samps_per_clk"]

        self.synci(200)

    def body(self):
        res_ch = self.cfg["res_ch"]
        step_cycles = self.us2cycles(self.cfg["step_hold_us"])

        self.trigger(pins=[0])
        self.pulse(ch=res_ch, t='auto')
        self.sync_all(step_cycles)

    def update(self):
        self.mathi(self.r_rp, self.r_addr, self.r_addr, '+', self.addr_step)

    def make_program(self):
        p = self
        rcount = 13
        rii = 14
        rjj = 15

        p.initialize()
        p.regwi(0, rcount, 0)
        p.regwi(0, rii, self.cfg['expts'] - 1)
        p.label("LOOP_I")
        p.regwi(0, rjj, self.cfg['reps'] - 1)
        p.label("LOOP_J")
        p.body()
        p.mathi(0, rcount, rcount, "+", 1)
        p.memwi(0, rcount, self.COUNTER_ADDR)
        p.loopnz(0, rjj, 'LOOP_J')
        p.update()
        p.loopnz(0, rii, "LOOP_I")

        res_ch = self.cfg["res_ch"]
        p.set_pulse_registers(
            ch=res_ch, style="arb", freq=0, phase=0, gain=self.cfg["gain"],
            waveform="trap_wfm", outsel="input", mode="periodic",
        )
        p.trigger(pins=[0])
        p.pulse(ch=res_ch, t='auto')
        p.end()

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch": GEN_CH,
    "reps": 1,
    "expts": NUM_STEPS,
    "idata_list": idata_list,
    "qdata_list": qdata_list,
    "trap_idata": trap_idata,
    "trap_qdata": trap_qdata,
    "samples_per_step": samples_per_step,
    "step_hold_us": STEP_HOLD_US,
    "gain": 32767,
}

prog = RepeatedStepSerrodyneProgram(soccfg, config)
prog.run(soc, start_src="external")
print(f"Running on hardware — {NUM_STEPS} sweep steps x {STEP_HOLD_US:.4f} us "
      f"= {NUM_STEPS*STEP_HOLD_US*1e-3:.4f} ms sweep, then trapping "
      f"(4 tones, periodic, indefinitely until soc.reset_gens()).")

Generator 1: f_fabric=614.400 MHz, samps_per_clk=16, envelope sample rate=9.8304 GSPS
Envelope memory available: 65536 samples
NUM_STEPS = 46
Fixed per-tone piece lengths (samples), used for every step + trap:
  tone 0 (+76.2 MHz offset): ratio 0.337 -> 384 samples (24.0 fabric cycles)
  tone 1 (-0.0 MHz offset): ratio 0.167 -> 224 samples (14.0 fabric cycles)
  tone 2 (-46.7 MHz offset): ratio 0.288 -> 416 samples (26.0 fabric cycles)
  tone 3 (-71.6 MHz offset): ratio 0.208 -> 272 samples (17.0 fabric cycles)
Composite buffer length (fixed, all steps + trap): 1296 samples
Composite buffer period T_buf = 131.8359 ns
K (periods held per chirp step) = 989
STEP_HOLD_US = 130.3857 us (target was 130.4348 us, snap delta = -49.040 ns)
Actual total sweep time = 5.9977 ms (target was 6.000 ms)
Note: sync_all()/us2cycles() still quantizes this wait to the tProc's own clock (a separate, generally much finer clock domain than the DAC sample clock), so there is a small residual rounding error on 

In [ ]:
soc.reset_gens()

In [5]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 1
RO_CH = 0

gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
# -----------------------------------------------------------------------------
def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0):
    n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

def serrodyne_tone_n(freq_hz, n_samples, sample_rate, amplitude, phase0=0.0, width=1.0):
    """Same as serrodyne_tone but takes an explicit sample count. Used to
    align the final tone in a composite buffer so it ends exactly on a
    whole number of periods (i.e. right at its peak), avoiding an extra
    arbitrary phase jump on top of the sawtooth's own reset when the
    buffer loops via mode='periodic'."""
    n_samples = max(1, int(n_samples))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION -- each step's buffer is a 4-tone composite
#    (ratio-weighted durations). All 4 tones shift together as the chirp
#    offset sweeps from CHIRP_OFFSET_START_HZ to CHIRP_OFFSET_STOP_HZ.
# -----------------------------------------------------------------------------
CHIRP_SPAN_HZ = 350e6
CHIRP_OFFSET_STOP_HZ = 0.0                                       # ends AT the base tones
CHIRP_OFFSET_START_HZ = CHIRP_OFFSET_STOP_HZ - CHIRP_SPAN_HZ      # starts 350 MHz below them

NUM_STEPS = 45
AMPLITUDE = 0.12 #0.11

BASE_TONES_HZ = np.array([-76.25e6, 0, +122.92e6-76.25e6, +147.82e6-76.25e6])
# BASE_TONES_HZ -= 100e6
TONE_RATIOS = np.array([0.337, 0.167, 0.288, 0.208])
# TONE_RATIOS = np.array([1,1,1,1])

# BASE_TONES_HZ = np.array([0])
# TONE_RATIOS = np.array([1.0])


BASE_TONES_HZ *= -1
NUM_TONES = len(BASE_TONES_HZ)


TOTAL_SWEEP_S = 6e-3 # 6e-3 
STEP_HOLD_S = TOTAL_SWEEP_S / NUM_STEPS
STEP_HOLD_US = STEP_HOLD_S * 1e6

max_feasible_total_s = 0.95 * ENV_MAXLEN / ENV_SR
CYCLE_S = max_feasible_total_s / (NUM_STEPS + 1)   # one composite buffer's total duration

tone_fractions = TONE_RATIOS / TONE_RATIOS.sum()
tone_durations_s = tone_fractions * CYCLE_S
print(f"Composite buffer cycle: {CYCLE_S*1e9:.2f} ns, held via mode='periodic' "
      f"for {STEP_HOLD_US:.2f} us per chirp step ({TOTAL_SWEEP_S*1e3:.3f} ms total sweep)")
for i, (bt, dur, r) in enumerate(zip(BASE_TONES_HZ, tone_durations_s, TONE_RATIOS)):
    print(f"  tone {i} ({bt/1e6:+.1f} MHz offset): ratio {r} -> requested {dur*1e9:.2f} ns")

def _peak_aligned_samples(freq_hz, target_duration_s, sample_rate, samps_per_clk):
    """Sample count nearest target_duration_s that lands the ramp exactly
    at its peak (an integer number of periods), rounded to a samps_per_clk
    multiple so no zero-padding is needed afterward. Falls back to a plain
    rounded/truncated count if freq ~ 0 (no periodicity to align to)."""
    if abs(freq_hz) < 1.0:
        n = int(round(target_duration_s * sample_rate))
        return max(samps_per_clk, (n // samps_per_clk) * samps_per_clk)
    period_samples = sample_rate / abs(freq_hz)
    target_n = target_duration_s * sample_rate
    n_periods = max(1, round(target_n / period_samples))
    ideal_n = n_periods * period_samples
    n = int(round(ideal_n / samps_per_clk)) * samps_per_clk
    return max(samps_per_clk, n)

def build_multitone_buffer(chirp_offset_hz, phase0):
    """One composite envelope: all NUM_TONES tones, each shifted by the same
    chirp_offset_hz and ratio-weighted in duration, concatenated. The final
    tone is snapped to a whole number of its own periods so the buffer's
    loop-back point (mode='periodic') coincides with a natural sawtooth
    reset instead of adding an extra arbitrary phase jump."""
    i_pieces, q_pieces = [], []
    phase = phase0
    n_tones = len(BASE_TONES_HZ)
    piece_lens = []
    for idx, (base_tone, dur) in enumerate(zip(BASE_TONES_HZ, tone_durations_s)):
        f = chirp_offset_hz + base_tone
        is_last = (idx == n_tones - 1)
        if is_last:
            n_samples = _peak_aligned_samples(f, dur, ENV_SR, samps_per_clk)
            y, phase, n_samples = serrodyne_tone_n(f, n_samples, ENV_SR, amplitude=AMPLITUDE, phase0=phase)
        else:
            y, phase, n_samples = serrodyne_tone(f, dur, ENV_SR, amplitude=AMPLITUDE, phase0=phase)
            pad = (-len(y)) % samps_per_clk
            if pad:
                y = np.concatenate([y, np.zeros(pad)])
        i_pieces.append(np.round(y * maxv).astype(np.int16))
        q_pieces.append(np.zeros(len(y), dtype=np.int16))
        piece_lens.append(len(i_pieces[-1]))
    return np.concatenate(i_pieces), np.concatenate(q_pieces), phase, piece_lens

chirp_offsets_hz = np.linspace(CHIRP_OFFSET_START_HZ, CHIRP_OFFSET_STOP_HZ, NUM_STEPS)
maxv = soccfg.get_maxv(GEN_CH)

idata_list = []
qdata_list = []
phase = 0.0
for offset in chirp_offsets_hz:
    idata, qdata, phase, piece_lens = build_multitone_buffer(offset, phase)
    idata_list.append(idata)
    qdata_list.append(qdata)

samples_per_step = len(idata_list[0])
total_samples = samples_per_step * NUM_STEPS
print(f"Per-step composite buffer length: {samples_per_step} samples "
      f"(tone slices: {piece_lens}, smallest = {min(piece_lens)/samps_per_clk:.1f} fabric cycles)")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
assert total_samples <= ENV_MAXLEN
assert samples_per_step % samps_per_clk == 0
assert min(piece_lens) >= 3 * samps_per_clk, (
    f"Smallest tone slice is only {min(piece_lens)} samples "
    f"({min(piece_lens)/samps_per_clk:.1f} fabric cycles) -- hardware needs >= 3. "
    f"Reduce NUM_STEPS, or make TONE_RATIOS less extreme."
)

# -----------------------------------------------------------------------------
# 3b. TRAP BUFFER: same ratio-weighted 4-tone composite, at the FINAL chirp
#     frequency, phase-continuous with the last sweep step.
# -----------------------------------------------------------------------------
trap_idata, trap_qdata, phase, trap_piece_lens = build_multitone_buffer(CHIRP_OFFSET_STOP_HZ, phase)
print(f"Trap buffer: {len(trap_idata)} samples (tone slices: {trap_piece_lens})")

total_with_trap = total_samples + len(trap_idata)
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")
assert total_with_trap <= ENV_MAXLEN, (
    f"Sweep + trap buffers exceed memory: need {total_with_trap}, "
    f"have {ENV_MAXLEN}. Reduce NUM_STEPS or CYCLE_S."
)

# -----------------------------------------------------------------------------
# 4. PROGRAM: sweep as before, then ONE final pulse on the concatenated trap
#    buffer with mode="periodic" -- hardware loops it forever on its own.
# -----------------------------------------------------------------------------
class RepeatedStepSerrodyneProgram(RAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for i, (idata_step, qdata_step) in enumerate(zip(cfg["idata_list"], cfg["qdata_list"])):
            self.add_envelope(ch=res_ch, name=f"serr_{i}", idata=idata_step, qdata=qdata_step)

        self.add_envelope(ch=res_ch, name="trap_wfm", idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])

        self.set_pulse_registers(
            ch=res_ch,
            style="arb",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            waveform="serr_0",
            outsel="input",
            mode="periodic",
        )

        self.r_rp = self.ch_page(res_ch)
        self.r_addr = self.sreg(res_ch, "addr")
        self.addr_step = cfg["samples_per_step"] // self.soccfg["gens"][res_ch]["samps_per_clk"]

        self.synci(200)

    def body(self):
        res_ch = self.cfg["res_ch"]
        step_cycles = self.us2cycles(self.cfg["step_hold_us"])

        self.trigger(pins=[0])
        self.pulse(ch=res_ch, t='auto')
        self.sync_all(step_cycles)

    def update(self):
        self.mathi(self.r_rp, self.r_addr, self.r_addr, '+', self.addr_step)

    def make_program(self):
        """Standard RAveragerProgram sweep (expts x reps), with a single
        trapping pulse appended after the loop -- no loop construct needed
        for trapping, since mode="periodic" makes the hardware repeat it
        on its own once triggered."""
        p = self
        rcount = 13
        rii = 14
        rjj = 15

        p.initialize()
        p.regwi(0, rcount, 0)
        p.regwi(0, rii, self.cfg['expts'] - 1)
        p.label("LOOP_I")
        p.regwi(0, rjj, self.cfg['reps'] - 1)
        p.label("LOOP_J")
        p.body()
        p.mathi(0, rcount, rcount, "+", 1)
        p.memwi(0, rcount, self.COUNTER_ADDR)
        p.loopnz(0, rjj, 'LOOP_J')
        p.update()
        p.loopnz(0, rii, "LOOP_I")

        # --- trapping: one pulse, periodic buffer, then end(). The DAC keeps
        # cycling the 4 concatenated tones forever regardless of what the
        # tProc does after this -- including after it hits end() and halts.
        res_ch = self.cfg["res_ch"]
        p.set_pulse_registers(
            ch=res_ch, style="arb", freq=0, phase=0, gain=self.cfg["gain"],
            waveform="trap_wfm", outsel="input", mode="periodic",
        )
        p.trigger(pins=[0])
        p.pulse(ch=res_ch, t='auto')
        p.end()

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch": GEN_CH,
    "reps": 1,
    "expts": NUM_STEPS,
    "idata_list": idata_list,
    "qdata_list": qdata_list,
    "trap_idata": trap_idata,
    "trap_qdata": trap_qdata,
    "samples_per_step": samples_per_step,
    "step_hold_us": STEP_HOLD_US,
    "gain": 32767,
}

prog = RepeatedStepSerrodyneProgram(soccfg, config)
prog.run(soc) #, start_src="external"
print(f"Running on hardware — {NUM_STEPS} sweep steps x {STEP_HOLD_US:.2f} us "
      f"= {NUM_STEPS*STEP_HOLD_US*1e-3:.3f} ms sweep, then trapping "
      f"(4 tones, periodic, indefinitely until soc.reset_gens()).")

Generator 1: f_fabric=614.400 MHz, samps_per_clk=16, envelope sample rate=9.8304 GSPS
Envelope memory available: 65536 samples
Composite buffer cycle: 137.68 ns, held via mode='periodic' for 133.33 us per chirp step (6.000 ms total sweep)
  tone 0 (+76.2 MHz offset): ratio 0.337 -> requested 46.40 ns
  tone 1 (-0.0 MHz offset): ratio 0.167 -> requested 22.99 ns
  tone 2 (-46.7 MHz offset): ratio 0.288 -> requested 39.65 ns
  tone 3 (-71.6 MHz offset): ratio 0.208 -> requested 28.64 ns
Per-step composite buffer length: 1376 samples (tone slices: [464, 240, 400, 272], smallest = 15.0 fabric cycles)
Total envelope samples (sweep): 61920 / 65536 available
Trap buffer: 1376 samples (tone slices: [464, 240, 400, 272])
Total envelope samples (sweep + trap): 63296 / 65536 available
Running on hardware — 45 sweep steps x 133.33 us = 6.000 ms sweep, then trapping (4 tones, periodic, indefinitely until soc.reset_gens()).
